# Exploring a Delzant polygon

Workflow: put in the vertices, then ask the `Polygon` for whatever you want. Each quantity is computed the first time you ask for it and cached; `P.computed()` shows what has been filled in so far.

Conventions (normalizations of S, dσ, A_ext, …) are in `CONVENTIONS.md`. In short: u = ½ Σ ℓᵢ log ℓᵢ + h, S = −½ Σ ∂ᵢ∂ⱼ uⁱʲ, L_A(f) = ∫_∂P f dσ − ∫_P A f dμ.

In [ ]:
%matplotlib inline
from fractions import Fraction
import numpy as np
from IPython.display import Markdown, display

from extremal import *            # Polygon, ToricMetric, x, y, projective_plane, hirzebruch, ...
from extremal import viz
from extremal.models import wang_zhou_nonagon
from extremal.stability import CreaseProfile, crease_search

## 1. The polygon

Vertices must be exact: ints, `Fraction`s, or strings like `'1/2'`. Either orientation is fine. This one is CP² blown up at one point (equivalently the Hirzebruch surface F₁).

In [ ]:
P = Polygon([(1, 0), (3, 0), (0, 3), (0, 1)], name="CP² blown up at a point")
P.computed()   # nothing computed yet

In [ ]:
display(Markdown(P.report()))
P.computed()

In [ ]:
viz.plot_polygon(P);

## 2. Exact invariants

A_ext is the affine function with L_{A_ext}(f) = 0 for every affine f; the Futaki invariant is F(f) = L_Ŝ(f). Both are exact rationals. The dashed lines are level sets of A_ext.

In [ ]:
print("S_hat  =", P.S_hat)
print("A_ext  =", P.A_ext)
print("Futaki =", P.futaki)
print("check: L(1), L(x), L(y) =", P.L(1), P.L(x), P.L(y))

ax = viz.plot_polygon(P, labels=False)
viz.plot_A_ext(P, ax=ax);

`L` accepts any polynomial in `x, y` (exact), with weight `A` defaulting to A_ext. For example L_{A_ext}(x² + y²) is positive, as it must be for a strictly convex f if P is to be stable:

In [ ]:
P.L(x**2 + y**2), P.L(x**2 + y**2, A=P.S_hat)

## 3. A metric and its curvature

`ToricMetric(P, h)` is the metric with symplectic potential u = u_G + h (h = 0 is Guillemin's metric). Curvature fields are vectorized functions of the momentum coordinates (x, y).

In [ ]:
g = ToricMetric(P)
pts = np.array([[1.0, 1.0], [2.0, 0.5], [0.5, 2.0], [0.5, 0.5]])   # the last point is on ∂P -> nan
print("S at points:     ", g.scalar_curvature(pts[:, 0], pts[:, 1]))
print("|Ric|^2 at points:", g.ricci_norm_sq(pts[:, 0], pts[:, 1]))
print("available fields:", list(g.FIELDS))

In [ ]:
viz.plot_field(g, "S");

Several fields side by side. Signed quantities (S − Ŝ, S − A_ext) use a diverging scale centred at 0; each panel has its own colour scale.

In [ ]:
viz.plot_fields(g);

### The Futaki kernel

For *any* metric, F(f) = ∫_P (S − Ŝ) f dμ. Check it numerically against the exact Futaki vector:

In [ ]:
Fx = integrate_numeric(P, lambda a, b: g.futaki_kernel(a, b) * a, order=40)
Fy = integrate_numeric(P, lambda a, b: g.futaki_kernel(a, b) * b, order=40)
print("numeric:", Fx, Fy)
print("exact:  ", float(P.futaki[0]), float(P.futaki[1]))

### Changing the potential

Add a smooth `h`. u must stay convex: check the smallest eigenvalue of Hess u over the polygon first.

In [ ]:
g2 = ToricMetric(P, h=x**3 / 30 + y**2 / 10)
mesh = viz.sample_mesh(P)
print("min eigenvalue of Hess u:", g2.min_hessian_eigenvalue(mesh.x, mesh.y).min())
viz.plot_fields(g2, names=("S", "S - A_ext", "|Ric|^2"));

## 4. Sanity check: Fubini–Study

On the standard simplex, Guillemin's metric is Fubini–Study: S ≡ 6 and |Ric|² ≡ 36 (Ric = 3g).

In [ ]:
fs = ToricMetric(projective_plane(1))
viz.plot_fields(fs, names=("S", "|Ric|^2", "|Ric_0|^2"));

## 5. Stability: crease functions

A **simple crease** is f = max(0, n·x − s): convex, piecewise linear. If L_{A_ext}(f) < 0 then P is relatively K-unstable and has no extremal metric.

**Why creases are enough (dimension 2).** Wang–Zhou (Adv. Math. 2011; Cor. 3.5 / Thm 3.4 in B. Zhou's thesis): if L_A(u) < 0 for *some* convex u, then L_A(f) < 0 for some simple crease f, with **no sign condition on A** (Donaldson 2002, Prop. 5.3.1, had the borderline case for A > 0). Donaldson (2002, Prop. 6.1): the crease can be taken rational. So checking rational creases decides whether L_{A_ext} ≥ 0 on all convex functions.

**Exact one-variable reduction.** For a fixed direction n, L(s) = ∫_s^{t_max} (t − s) dρ(t), where ρ has density (boundary crossings) Σ Lᵢ/|Δtᵢ| minus (A_ext integrated along the chord {n·x = t})/|n|. ρ is quadratic between vertex values, so L is piecewise quartic in s and is minimized exactly. `CreaseProfile` builds this; it is exact for rational n and s.

In [ ]:
viz.plot_crease(P, 1, 1, -2);

In [ ]:
import matplotlib.pyplot as plt

prof = CreaseProfile(P, (1, 1))
print("L at s = 2 (exact):", prof.L(2), "  direct:", P.L_crease(1, 1, -2))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
viz.plot_crease_profile(prof, ax=axes[0])
viz.plot_crease_profile(prof, ax=axes[1], normalized=True)
fig.tight_layout()

### Searching all directions

L is homogeneous in (n, s) and n, −n give the same values, so the creases form a half-circle of directions θ times one offset. For each θ, `crease_search` takes the scale-invariant **margin** min_s L(f) / ∫_∂P f dσ (f normalized to vanish at the barycenter, as in Donaldson's uniform stability). If the margin goes negative, it finds a nearby *rational* crease and checks L < 0 exactly. Floats only locate candidates; a positive result is evidence, not proof.

Calabi constructed an extremal metric on this P, so the margin should be positive everywhere.

In [ ]:
result = crease_search(P)
print(result)
viz.plot_crease_margin(result);

### An unstable example: the Wang–Zhou nonagon

B. Zhou's thesis (§3.2.2) gives a 9-vertex Delzant polygon P* depending on a parameter α, relatively K-unstable for α large. The search finds an exact rational destabilizing crease:

In [ ]:
Q = wang_zhou_nonagon(10_000)
rq = crease_search(Q)
print(rq)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={'width_ratios': [2, 1]})
viz.plot_crease_margin(rq, ax=axes[0])
viz.plot_crease(Q, *rq.certificate, ax=axes[1])
axes[1].set_aspect('auto')   # the polygon is 7 wide and ~20000 tall: zoom on the top
axes[1].set_xlim(-0.5, 7.5); axes[1].set_ylim(9750, 10040)
axes[1].set_title('top of P* with the destabilizing crease', fontsize=10, loc='left')
fig.tight_layout()

Where does instability set in? (Bisection gives a threshold between α ≈ 6854 and 6882.)

In [ ]:
for alpha in [100, 1000, 3000, 6000, 6800, 6900, 8000, 10_000]:
    r = crease_search(wang_zhou_nonagon(alpha), n_angles=180)
    print(f"α = {alpha:>6}:  margin = {r.best_margin:+.3e}   {'UNSTABLE (exact)' if r.unstable else ''}")

### Horizontal versus tilted creases

Zhou's argument uses horizontal creases max(0, y − α + k) with α ≫ k ≫ 1: their value tends to C − k/7 as α → ∞ (slope −1/7 = 27 − 27⅐ from the thesis; numerically C ≈ 72.48, matching 1522/21). So k must exceed about 500 before they destabilize, and α must be much larger than k. Tilted creases destabilize far earlier.

In [ ]:
from fractions import Fraction

ks = np.linspace(1, 3000, 240)
far = CreaseProfile(wang_zhou_nonagon(10**10), (0, 1))   # effectively the α → ∞ limit
C = float(far.L(Fraction(10**10 - 10)) + Fraction(10, 7))

fig, ax = plt.subplots(figsize=(8, 4))
for alpha, color in zip([10_000, 100_000, 1_000_000], ['#2a78d6', '#eb6834', '#1baf7a']):
    prof = CreaseProfile(wang_zhou_nonagon(alpha), (0, 1))
    ax.plot(ks, prof.L_values(alpha - ks), color=color, lw=2, label=f'α = {alpha:,}')
ax.plot(ks, C - ks / 7, color='#52514e', lw=1.2, ls='--', label=f'limit  {C:.2f} − k/7')
ax.axhline(0, color='#52514e', lw=0.8)
ax.set_xlabel('k  (crease y = α − k)')
ax.set_title('L_{A_ext}(max(0, y − α + k)) on the Wang–Zhou nonagon', loc='left', fontsize=10)
ax.legend(frameon=False, fontsize=9)
ax.grid(True, color='#e4e3df', lw=0.6)
for side in ('top', 'right'): ax.spines[side].set_visible(False)

Same comparison through the normalized margin: θ = 90° is the horizontal direction, against the worst direction overall. Thresholds by bisection: tilted creases destabilize from α ≈ 6870, horizontal ones only from α ≈ 51600.

In [ ]:
print(f"{'α':>8} | {'horizontal margin':>18} | {'best margin':>12} | {'best θ':>7} | exact certificate")
for alpha in [6_000, 7_000, 10_000, 50_000, 53_000, 100_000]:
    r = crease_search(wang_zhou_nonagon(alpha), n_angles=180)
    horizontal = r.margin[90]          # theta[90] = 90°, normal (0, 1)
    print(f"{alpha:>8} | {horizontal:>+18.3e} | {r.best_margin:>+12.3e} | {np.degrees(r.best_theta):>6.1f}° | "
          f"{'yes' if r.unstable else 'no'}")

## 6. Random Delzant polygons

Built from CP² or F_n by corner cuts (toric blow-ups); the history records how. Change the seeds to explore.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, seed in zip(axes.ravel(), range(6)):
    Q, history = random_delzant(n_blowups=3, seed=seed)
    viz.plot_field(ToricMetric(Q), "S - A_ext", ax=ax, resolution=80)
    ax.set_title(f"seed {seed}: " + ", ".join(history[:1]) + f" + {len(history) - 1} cuts", fontsize=9, loc="left")
fig.tight_layout()

In [ ]:
Q, history = random_delzant(n_blowups=3, seed=4)
print("\n".join(history))
display(Markdown(Q.report()))
print(crease_search(Q))